In [1]:
import torch
import torch.nn as nn

In [2]:
class Discriminator(nn.Module):
  def __init__(self,channels_img,features_d):
    super().__init__()
    self.disc=nn.Sequential(
     nn.Conv2d(channels_img,features_d,kernel_size=4,stride=2,padding=1),
     nn.LeakyReLU(0.2),
     self._block(features_d,features_d*2,4,2,1),
     self._block(features_d*2,features_d*4,4,2,1),
     self._block(features_d*4,features_d*8,4,2,1),
     nn.Conv2d(features_d*8,1,kernel_size=4,stride=1,padding=0),
     )
  def _block(self,in_channels,out_channels,kernel_size,stride,padding):
    return nn.Sequential(
        nn.Conv2d(in_channels,out_channels,
        kernel_size,stride,padding,bias=False
    ),
        nn.LeakyReLU(0.2),)
  def forward(self,x):
    return self.disc(x)

In [3]:
class Generator(nn.Module):
  def __init__(self,channles_noise,channles_img,features_g):
    super().__init__()
    self.net=nn.Sequential(
        self._block(channles_noise,features_g*16,4,1,0),
        self._block(features_g*16,features_g*8,4,2,1),
        self._block(features_g*8,features_g*4,4,2,1),
        self._block(features_g*4,features_g*2,4,2,1),
        nn.ConvTranspose2d(features_g*2,channles_img,kernel_size=4,stride=2,padding=1),
        nn.Tanh()
    )
  def _block(self,in_channels,out_channels,kernel_size,stride,padding):
    return nn.Sequential(
        nn.ConvTranspose2d(
            in_channels,out_channels,kernel_size,stride,padding,
            bias=False
        ),
        nn.BatchNorm2d(out_channels),
        nn.ReLU()
    )
  def forward(self,x):
    return self.net(x)

In [4]:
def inital_weights(model):
  for m in model.modules():
    if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d,nn.BatchNorm2d)):
      nn.init.normal_(m.weight.data,0.0,0.02)

In [5]:
def test():
  N,in_channels,H,W=8,3,64,64
  noise_dim=100
  x=torch.randn((N,in_channels,H,W))
  disc=Discriminator(in_channels,8)
  assert disc(x).shape==(N,1,1,1),'Test Failed discriminator'
  gen=Generator(noise_dim,in_channels,8)
  z=torch.randn((N,noise_dim,1,1))
  assert gen(z).shape==(N,in_channels,H,W),'Test Failed generator'

In [6]:
test()

In [7]:
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

In [8]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
lr=2e-4
batch=128
img_size=64
channels_img=1
noise_dim=100
epochs=5
features_disc=64
features_gen=64

In [10]:
transforms=transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(channels_img)],
     [0.5 for _ in range(channels_img)])
])

In [11]:
dataset=datasets.MNIST(root='dataset/',train=True,transform=transforms,download=True)

In [12]:
dataloader=DataLoader(dataset,batch_size=batch,shuffle=True)

In [13]:
gen=Generator(noise_dim,channels_img,features_gen).to(device)
disc=Discriminator(channels_img,features_disc).to(device)

In [14]:
inital_weights(gen)
inital_weights(disc)

In [15]:
opt_gen=optim.Adam(gen.parameters(),lr=lr,betas=(0.5,0.999))
opt_disc=optim.Adam(disc.parameters(),lr=lr,betas=(0.5,0.999))

In [17]:
loss=nn.BCEWithLogitsLoss()

In [18]:
fixed_noise=torch.randn(32,noise_dim,1,1).to(device)
writer_real=SummaryWriter(f'logs/real')
writer_fake=SummaryWriter(f'logs/fake')
step=0

In [19]:
gen.train()
disc.train()

Discriminator(
  (disc): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.2)
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): LeakyReLU(negative_slope=0.2)
    )
    (3): Sequential(
      (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): LeakyReLU(negative_slope=0.2)
    )
    (4): Sequential(
      (0): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): LeakyReLU(negative_slope=0.2)
    )
    (5): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1))
  )
)

In [20]:
for epoch in range(epochs):
    for batch_idx, (real, _) in enumerate(dataloader):
        real = real.to(device)
        noise = torch.randn(batch, noise_dim, 1, 1).to(device)
        fake = gen(noise)
        disc_real = disc(real).reshape(-1)
        loss_disc_real = loss(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake.detach()).reshape(-1)
        loss_disc_fake = loss(disc_fake, torch.zeros_like(disc_fake))
        loss_disc = (loss_disc_real + loss_disc_fake) / 2
        disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()
        output = disc(fake).reshape(-1)
        loss_gen = loss(output, torch.ones_like(output))
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()
        if batch_idx % 100 == 0:
            print(
                f"Epoch [{epoch}/{epochs}] Batch {batch_idx}/{len(dataloader)} \
                  Loss D: {loss_disc:.4f}, loss G: {loss_gen:.4f}"
            )

            with torch.no_grad():
                fake = gen(fixed_noise)
                img_grid_real = torchvision.utils.make_grid(real[:32], normalize=True)
                img_grid_fake = torchvision.utils.make_grid(fake[:32], normalize=True)

                writer_real.add_image("Real", img_grid_real, global_step=step)
                writer_fake.add_image("Fake", img_grid_fake, global_step=step)

            step += 1

Epoch [0/5] Batch 0/469                   Loss D: 0.6900, loss G: 0.7525
Epoch [0/5] Batch 100/469                   Loss D: 0.0150, loss G: 5.5806
Epoch [0/5] Batch 200/469                   Loss D: 0.0071, loss G: 5.6906
Epoch [0/5] Batch 300/469                   Loss D: 0.0941, loss G: 4.2702
Epoch [0/5] Batch 400/469                   Loss D: 0.1048, loss G: 3.0388
Epoch [1/5] Batch 0/469                   Loss D: 0.2147, loss G: 2.3153
Epoch [1/5] Batch 100/469                   Loss D: 0.2065, loss G: 2.8749
Epoch [1/5] Batch 200/469                   Loss D: 0.1671, loss G: 2.6101
Epoch [1/5] Batch 300/469                   Loss D: 0.3320, loss G: 2.1769
Epoch [1/5] Batch 400/469                   Loss D: 0.7414, loss G: 1.4175
Epoch [2/5] Batch 0/469                   Loss D: 0.6412, loss G: 1.6509
Epoch [2/5] Batch 100/469                   Loss D: 0.5928, loss G: 1.2927
Epoch [2/5] Batch 200/469                   Loss D: 0.5214, loss G: 0.9478
Epoch [2/5] Batch 300/469      